# Chapter 24 — Recurrent Neural Networks II: Embeddings, Seq2Seq & Evaluation Metrics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch24_recurrent_neural_networks_ii/ch24_notebook.ipynb)

Companion notebook for **Chapter 24** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- The encoder-decoder paradigm maps one sequence to another: one network reads, another writes.
- Word embeddings turn words into dense vectors where words used in similar contexts sit close together.
- Count-based methods use global co-occurrence statistics; Word2Vec learns by prediction; GloVe combines both ideas.
- Negative sampling makes Skip-Gram affordable by replacing a huge softmax with a few cheap binary decisions.
- Generated text is scored against references: BLEU for precision, METEOR for recall and synonyms, ROUGE for summaries, CIDEr for captions.

## Example 1: Count-Based Word Embedding (Co-occurrence to SVD)

This builds word vectors the count-based way: build a co-occurrence matrix from a tiny corpus, convert it to Positive PMI, then apply SVD to get compact 2-D vectors. The cosine similarities confirm that words used in similar contexts ('cat' and 'dog') end up with similar vectors. This captures global co-occurrence statistics.

In [1]:
# Chapter 24 - Example 1: Count-based word embedding (co-occurrence -> PMI -> SVD)
import numpy as np
np.random.seed(0)

print("=== Count-Based Word Embedding: Co-occurrence -> PPMI -> SVD ===\n")

# A tiny corpus
corpus = ("the cat sat on the mat . the dog sat on the rug . "
          "the cat ran . the dog ran . the cat and the dog play").split()
print("Corpus:", " ".join(corpus), "\n")

# Build vocabulary
vocab = sorted(set(corpus))
V = len(vocab)
idx = {w: i for i, w in enumerate(vocab)}
print(f"Vocabulary ({V} words): {vocab}\n")

# Co-occurrence matrix with context window = 1 (neighbors)
C = np.zeros((V, V))
window = 1
for i, w in enumerate(corpus):
    for j in range(max(0, i-window), min(len(corpus), i+window+1)):
        if i != j:
            C[idx[w], idx[corpus[j]]] += 1

print("Co-occurrence counts computed (window=1).")

# Positive Pointwise Mutual Information (PPMI)
total = C.sum()
row_sum = C.sum(axis=1, keepdims=True)
col_sum = C.sum(axis=0, keepdims=True)
with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.log2((C * total) / (row_sum @ col_sum))
    pmi[np.isnan(pmi)] = 0
    pmi[np.isinf(pmi)] = 0
ppmi = np.maximum(pmi, 0)   # positive PMI only
print("PPMI matrix computed (negative values clipped to 0).\n")

# SVD to get low-dimensional embeddings
U, S, Vt = np.linalg.svd(ppmi)
r = 2   # embedding dimension
embeddings = U[:, :r] * S[:r]   # W = U_hat * S_hat
print(f"SVD applied. Keeping top r={r} dimensions.")
print(f"PPMI matrix size: {ppmi.shape} -> Word embeddings: {embeddings.shape}\n")

print("2-D word embeddings (first 6 words):")
for w in vocab[:6]:
    vec = embeddings[idx[w]]
    print(f"  {w:>6}: [{vec[0]:+.3f}, {vec[1]:+.3f}]")

# Cosine similarity between related words
def cossim(a, b):
    return a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)

print("\nCosine similarity between word pairs:")
for w1, w2 in [('cat','dog'), ('sat','ran'), ('cat','mat')]:
    s = cossim(embeddings[idx[w1]], embeddings[idx[w2]])
    print(f"  sim({w1}, {w2}) = {s:+.3f}")
print("\n'cat' and 'dog' appear in similar contexts -> similar vectors.")
print("Count-based methods capture GLOBAL co-occurrence statistics.")

=== Count-Based Word Embedding: Co-occurrence -> PPMI -> SVD ===

Corpus: the cat sat on the mat . the dog sat on the rug . the cat ran . the dog ran . the cat and the dog play 

Vocabulary (11 words): ['.', 'and', 'cat', 'dog', 'mat', 'on', 'play', 'ran', 'rug', 'sat', 'the']

Co-occurrence counts computed (window=1).
PPMI matrix computed (negative values clipped to 0).

SVD applied. Keeping top r=2 dimensions.
PPMI matrix size: (11, 11) -> Word embeddings: (11, 2)

2-D word embeddings (first 6 words):
       .: [-1.482, -0.853]
     and: [-1.012, +0.447]
     cat: [-1.537, -1.275]
     dog: [-1.982, -2.284]
     mat: [-0.855, +0.152]
      on: [-1.257, -1.370]

Cosine similarity between word pairs:
  sim(cat, dog) = +0.987
  sim(sat, ran) = +0.994
  sim(cat, mat) = +0.646

'cat' and 'dog' appear in similar contexts -> similar vectors.
Count-based methods capture GLOBAL co-occurrence statistics.


## Example 2: Word2Vec Skip-Gram Embeddings

This trains a Skip-Gram model with plain SGD on a small corpus with three semantic groups (royalty, animals, fruits). After training, the nearest-neighbor words by cosine similarity cluster by meaning - the model learned semantics purely from context, with no labels. This is the prediction-based alternative to the count method.

In [2]:
# Chapter 24 - Example 2: Word2Vec Skip-Gram intuition with cosine similarity
import numpy as np
np.random.seed(42)

print("=== Word2Vec Skip-Gram: Learning Embeddings by Prediction ===\n")
print("Skip-Gram predicts CONTEXT words from a CENTER word.")
print("Words used in similar contexts end up with similar vectors.\n")

# Tiny corpus with clear semantic groups
sentences = [
    "king queen royal palace throne",
    "king queen royal crown throne",
    "dog cat pet animal furry",
    "dog cat pet animal cute",
    "apple banana fruit sweet juicy",
    "apple banana fruit sweet tasty",
]
tokens = [s.split() for s in sentences]
vocab = sorted({w for s in tokens for w in s})
V = len(vocab)
idx = {w: i for i, w in enumerate(vocab)}
print(f"Vocabulary ({V} words): {vocab}\n")

# Build (center, context) training pairs with window=2
pairs = []
window = 2
for sent in tokens:
    for i, center in enumerate(sent):
        for j in range(max(0,i-window), min(len(sent), i+window+1)):
            if i != j:
                pairs.append((idx[center], idx[sent[j]]))
print(f"Generated {len(pairs)} (center, context) training pairs.\n")

# Skip-gram with simple SGD (embedding dim = 8)
dim = 8
W_in  = np.random.randn(V, dim) * 0.1   # center word embeddings
W_out = np.random.randn(V, dim) * 0.1   # context word embeddings

def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()

lr = 0.05
for epoch in range(300):
    np.random.shuffle(pairs)
    for c, o in pairs:
        h = W_in[c]                      # center embedding
        scores = W_out @ h               # similarity to all words
        probs = softmax(scores)
        probs[o] -= 1                    # gradient of cross-entropy
        grad_out = np.outer(probs, h)
        grad_in  = W_out.T @ probs
        W_out -= lr * grad_out
        W_in[c] -= lr * grad_in

print("Training complete (300 epochs of SGD).\n")

def cossim(a, b):
    return a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)

def nearest(word, k=3):
    v = W_in[idx[word]]
    sims = [(w, cossim(v, W_in[idx[w]])) for w in vocab if w != word]
    return sorted(sims, key=lambda x: -x[1])[:k]

print("Nearest neighbors by cosine similarity (learned embeddings):")
for w in ['king', 'dog', 'apple']:
    neighbors = nearest(w)
    nbr_str = ", ".join(f"{n}({s:.2f})" for n, s in neighbors)
    print(f"  {w:>6} -> {nbr_str}")

print("\nWords from the same semantic group (royalty / animals / fruits)")
print("cluster together - the model learned meaning from context alone.")

=== Word2Vec Skip-Gram: Learning Embeddings by Prediction ===

Skip-Gram predicts CONTEXT words from a CENTER word.
Words used in similar contexts end up with similar vectors.

Vocabulary (18 words): ['animal', 'apple', 'banana', 'cat', 'crown', 'cute', 'dog', 'fruit', 'furry', 'juicy', 'king', 'palace', 'pet', 'queen', 'royal', 'sweet', 'tasty', 'throne']

Generated 84 (center, context) training pairs.

Training complete (300 epochs of SGD).

Nearest neighbors by cosine similarity (learned embeddings):
    king -> crown(0.64), palace(0.64), throne(0.30)
     dog -> animal(0.65), cute(0.31), furry(0.30)
   apple -> sweet(0.67), tasty(0.33), juicy(0.30)

Words from the same semantic group (royalty / animals / fruits)
cluster together - the model learned meaning from context alone.


## Example 3: BLEU Score from Scratch

This builds the BLEU metric used to score generated text against a reference: n-gram precision combined with a brevity penalty. A perfect match scores 1.0; a too-short output is penalized heavily; a reworded sentence scores partially. The summary contrasts BLEU with METEOR, ROUGE, and CIDEr.

In [3]:
# Chapter 24 - Example 3: BLEU score from scratch (text-generation evaluation)
import numpy as np
from collections import Counter

print("=== BLEU Score from Scratch (Machine Translation / Captioning metric) ===\n")
print("BLEU = precision-based n-gram matching between generated & reference text,")
print("multiplied by a brevity penalty for too-short outputs.\n")

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def modified_precision(cand, ref, n):
    cand_ng = Counter(ngrams(cand, n))
    ref_ng  = Counter(ngrams(ref, n))
    if not cand_ng:
        return 0.0
    clipped = sum(min(cnt, ref_ng[ng]) for ng, cnt in cand_ng.items())
    return clipped / sum(cand_ng.values())

def bleu(candidate, reference, max_n=2):
    cand, ref = candidate.split(), reference.split()
    print(f"  Candidate: '{candidate}'")
    print(f"  Reference: '{reference}'")
    precisions = []
    for n in range(1, max_n+1):
        p = modified_precision(cand, ref, n)
        precisions.append(p)
        print(f"    {n}-gram precision: {p:.4f}")
    geo_mean = (np.exp(np.mean([np.log(p) for p in precisions]))
                if min(precisions) > 0 else 0.0)
    c, r = len(cand), len(ref)
    bp = 1.0 if c > r else (np.exp(1 - r/c) if c > 0 else 0.0)
    score = bp * geo_mean
    print(f"    geo-mean={geo_mean:.4f}  brevity-penalty={bp:.4f} (c={c}, r={r})")
    print(f"    => BLEU-{max_n} = {score:.4f}\n")
    return score

print("--- Example 1: Perfect match (BLEU-4) ---")
bleu("the cat is on the mat", "the cat is on the mat", max_n=4)

print("--- Example 2: Partially correct (BLEU-2) ---")
bleu("the cat sat on mat", "the cat is on the mat", max_n=2)

print("--- Example 3: Too short - brevity penalty (BLEU-2) ---")
bleu("the cat", "the cat is on the mat", max_n=2)

print("--- Example 4: Good but reworded (BLEU-2) ---")
bleu("the cat is on the mat", "a cat sits on the mat", max_n=2)

print("="*58)
print("BLEU ranges 0..1. Perfect match = 1.0. Missing higher-order")
print("n-grams or too-short output sharply lowers the score.")
print("BLEU = precision-focused; METEOR adds recall + synonyms;")
print("ROUGE = recall-focused (summaries); CIDEr weights rare n-grams.")

=== BLEU Score from Scratch (Machine Translation / Captioning metric) ===

BLEU = precision-based n-gram matching between generated & reference text,
multiplied by a brevity penalty for too-short outputs.

--- Example 1: Perfect match (BLEU-4) ---
  Candidate: 'the cat is on the mat'
  Reference: 'the cat is on the mat'
    1-gram precision: 1.0000
    2-gram precision: 1.0000
    3-gram precision: 1.0000
    4-gram precision: 1.0000
    geo-mean=1.0000  brevity-penalty=1.0000 (c=6, r=6)
    => BLEU-4 = 1.0000

--- Example 2: Partially correct (BLEU-2) ---
  Candidate: 'the cat sat on mat'
  Reference: 'the cat is on the mat'
    1-gram precision: 0.8000
    2-gram precision: 0.2500
    geo-mean=0.4472  brevity-penalty=0.8187 (c=5, r=6)
    => BLEU-2 = 0.3661

--- Example 3: Too short - brevity penalty (BLEU-2) ---
  Candidate: 'the cat'
  Reference: 'the cat is on the mat'
    1-gram precision: 1.0000
    2-gram precision: 1.0000
    geo-mean=1.0000  brevity-penalty=0.1353 (c=2, r=6)
